In [2]:
import pandas as pd
import re
import threading
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
import os

# 🔥 FUNCIÓN NUEVA → DETECTA DOWNLOADS / DESCARGAS
def obtener_ruta_salida():
    home = os.path.expanduser("~")

    posibles = [
        os.path.join(home, "Downloads"),
        os.path.join(home, "downloads"),
        os.path.join(home, "Descargas"),
        os.path.join(home, "descargas"),
        os.path.join(home, "Desktop"),
        os.path.join(home, "Escritorio"),
    ]

    for ruta in posibles:
        if os.path.exists(ruta):
            return os.path.join(ruta, "SEKRE1_ESCOLA.xlsx")

    # fallback final
    return os.path.join(home, "SEKRE1_ESCOLA.xlsx")


def scrape_products():
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

    all_products_data = []
    today_date = datetime.now().strftime('%d-%m-%Y')

    sizes = ['20','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36','37','38','39','40']

    driver.get("https://www.skechers.cl/escolares")

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.item-producto a'))
        )
    except TimeoutException:
        print("No se encontraron productos.")
        driver.quit()
        return

    start_time = time.time()

    # 🔥 RUTA DEFINIDA UNA VEZ
    output_path = obtener_ruta_salida()
    print(f"Guardando en: {output_path}")

    while True:
        if time.time() - start_time > 60 * 5:
            print("Límite de tiempo alcanzado.")
            break

        product_links = driver.find_elements(By.CSS_SELECTOR, '.item-producto a')

        if not product_links:
            print("No hay más productos.")
            break

        processed_links = set()

        for index in range(len(product_links)):
            product_link = product_links[index]
            product_url = product_link.get_attribute('href')

            if product_url in processed_links:
                continue

            try:
                driver.execute_script("arguments[0].scrollIntoView(true);", product_link)
                time.sleep(1)
                product_link.click()

                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, 'h1'))
                )

                try:
                    original_price = driver.find_element(By.XPATH, "//s").text.replace('$','').replace('.','')
                except:
                    original_price = '0'

                try:
                    final_price = driver.find_element(By.CSS_SELECTOR, 'b.text-danger').text
                except:
                    final_price = ''

                try:
                    sku = driver.find_element(By.CSS_SELECTOR, 'p.nomargin.text-muted').text.split("Código: ")[1]
                except:
                    sku = ''

                stock_dict = {size: 0 for size in sizes}

                for size in sizes:
                    try:
                        size_link = driver.find_element(By.CSS_SELECTOR, f'a[data-talla="{size}"]')
                        stock_dict[size] = int(size_link.get_attribute('data-cantidad'))
                    except:
                        stock_dict[size] = 0

                try:
                    image_url = driver.find_element(By.XPATH, "//a[@class='thumbnail active']/img").get_attribute('src')
                except:
                    image_url = ''

                product_url = driver.current_url

                product_name = product_url.split("/")[-1].replace('-', ' ').title()

                try:
                    color = driver.find_element(By.CSS_SELECTOR, "a.tallas.thumbnail[data-color]").get_attribute('data-color')
                except:
                    color = ''

                product_data = {
                    'Nombre': product_name,
                    'Marca': 'Skechers',
                    'Color': color,
                    'SKU': sku,
                    'Precio Original': original_price,
                    'Precio Final': final_price,
                    'URL de la Imagen': image_url,
                    'URL Página': product_url,
                    'Fecha': today_date,
                }

                product_data.update(stock_dict)
                all_products_data.append(product_data)

                # 🔥 GUARDA SIEMPRE (sobrescribe sin problema)
                df = pd.DataFrame(all_products_data)
                df.to_excel(output_path, index=False)

                processed_links.add(product_url)

            except:
                driver.back()
                continue

            driver.back()
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.item-producto a'))
            )
            product_links = driver.find_elements(By.CSS_SELECTOR, '.item-producto a')

    driver.quit()


# THREAD
scraping_thread = threading.Thread(target=scrape_products)
scraping_thread.start()

print("Scraping corriendo en segundo plano...")

Scraping corriendo en segundo plano...
Guardando en: C:\Users\const\Downloads\SEKRE1_ESCOLA.xlsx
